 Setup & Load

In [16]:
import pandas as pd
import sqlite3

In [ ]:
# Colab mount fallback: set DATA_ROOT to either Colab Drive or local './'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/'
except Exception:
    DATA_ROOT = './'
print('DATA_ROOT set to', DATA_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load df_master with safe fallback and explicit filename in repo
csv_path = DATA_ROOT + 'df_master.csv'
if not os.path.exists(csv_path):
    # repo contains df_master-2.csv; prefer that if original not found
    csv_path = 'df_master-2.csv' if os.path.exists('df_master-2.csv') else 'df_master.csv'
df = pd.read_csv(csv_path)


load into SQLite

In [19]:
conn = sqlite3.connect(':memory:')
df.to_sql('f1', conn, index=False, if_exists='replace')


440

In [20]:
print("Loaded:", df.shape)
df.head(3)

Loaded: (440, 23)


,race_date,track,driver_name,driver_code,driver_id,driver_number,nationality,dob,team,grid_position,...,time_retired,time_type,points,set_fastest_lap,fastest_lap_time,status,length_km,opened,longitude,latitude
0,2023-03-05,Bahrain,Max Verstappen,VER,max_verstappen,1,Dutch,1997-09-30,Red Bull Racing Honda RBPT,1,...,1:33:56.736,winner_time,25,No,1:36.236,Finished,5412,2002,50.510539,26.031766
1,2023-03-05,Bahrain,Sergio Perez,PER,perez,11,Mexican,1990-01-26,Red Bull Racing Honda RBPT,2,...,+11.987,gap_to_winner,18,No,1:36.344,Finished,5412,2002,50.510539,26.031766
2,2023-03-05,Bahrain,Fernando Alonso,ALO,alonso,14,Spanish,1981-07-29,Aston Martin Aramco Mercedes,5,...,+38.637,gap_to_winner,15,No,1:36.156,Finished,5412,2002,50.510539,26.031766


Query 1: Total points per driver (simple aggregation)

In [21]:
q1 = pd.read_sql_query("""
    SELECT driver_name, nationality, team,
           SUM(points) AS total_points,
           COUNT(*) AS races_entered
    FROM f1
    GROUP BY driver_name, nationality, team
    ORDER BY total_points DESC
    LIMIT 10
""", conn)

print("=== Q1: Championship Standings ===")
print(q1.to_string(index=False))

=== Q1: Championship Standings ===
    driver_name nationality                         team  total_points  races_entered
 Max Verstappen       Dutch   Red Bull Racing Honda RBPT           530             22
   Sergio Perez     Mexican   Red Bull Racing Honda RBPT           260             22
 Lewis Hamilton     British                     Mercedes           217             22
Fernando Alonso     Spanish Aston Martin Aramco Mercedes           198             22
Charles Leclerc  Monegasque                      Ferrari           185             22
   Lando Norris     British             McLaren Mercedes           184             22
   Carlos Sainz     Spanish                      Ferrari           178             22
 George Russell     British                     Mercedes           157             22
  Oscar Piastri  Australian             McLaren Mercedes            82             22
   Lance Stroll    Canadian Aston Martin Aramco Mercedes            68             22


Query 2: DNF rate per team (filtering + aggregation)

In [22]:
q2 = pd.read_sql_query("""
    SELECT team,
           COUNT(*) AS total_entries,
           SUM(CASE WHEN status IN ('Retired','Accident','Collision damage') THEN 1 ELSE 0 END) AS dnfs,
           ROUND(100.0 * SUM(CASE WHEN status IN ('Retired','Accident','Collision damage') THEN 1 ELSE 0 END) / COUNT(*), 1) AS dnf_pct
    FROM f1
    GROUP BY team
    ORDER BY dnf_pct DESC
""", conn)

print("=== Q2: DNF Rate by Team ===")
print(q2.to_string(index=False))

=== Q2: DNF Rate by Team ===
                        team  total_entries  dnfs  dnf_pct
           Williams Mercedes             44    11     25.0
              Alpine Renault             44    10     22.7
                Haas Ferrari             44     7     15.9
          Alfa Romeo Ferrari             44     6     13.6
                    Mercedes             44     5     11.4
Aston Martin Aramco Mercedes             44     5     11.4
            McLaren Mercedes             44     4      9.1
                     Ferrari             44     4      9.1
       AlphaTauri Honda RBPT             44     4      9.1
  Red Bull Racing Honda RBPT             44     2      4.5


Query 3: Pole-to-win conversion (multi-condition join logic)

In [23]:
q3 = pd.read_sql_query("""
    SELECT team,
           SUM(CASE WHEN grid_position = 1 THEN 1 ELSE 0 END) AS poles,
           SUM(CASE WHEN finish_position = 1 THEN 1 ELSE 0 END) AS wins,
           SUM(CASE WHEN grid_position = 1 AND finish_position = 1 THEN 1 ELSE 0 END) AS pole_to_win,
           ROUND(100.0 * SUM(CASE WHEN grid_position = 1 AND finish_position = 1 THEN 1 ELSE 0 END)
                 / NULLIF(SUM(CASE WHEN grid_position = 1 THEN 1 ELSE 0 END), 0), 1) AS conversion_pct
    FROM f1
    GROUP BY team
    HAVING poles > 0
    ORDER BY conversion_pct DESC
""", conn)

print("=== Q3: Pole-to-Win Conversion by Team ===")
print(q3.to_string(index=False))

=== Q3: Pole-to-Win Conversion by Team ===
                      team  poles  wins  pole_to_win  conversion_pct
Red Bull Racing Honda RBPT     14    21           13            92.9
                   Ferrari      7     1            1            14.3
                  Mercedes      1     0            0             0.0


 Query 4: Grid vs finish position gain/loss per driver (window function)

In [24]:
q4 = pd.read_sql_query("""
    SELECT driver_name,
           ROUND(AVG(grid_position), 2) AS avg_grid,
           ROUND(AVG(finish_position), 2) AS avg_finish,
           ROUND(AVG(grid_position - finish_position), 2) AS avg_positions_gained,
           COUNT(*) AS races
    FROM f1
    WHERE status = 'Finished'
    GROUP BY driver_name
    HAVING races >= 10
    ORDER BY avg_positions_gained DESC
""", conn)

print("=== Q4: Average Positions Gained Per Race (Finishers Only) ===")
print(q4.to_string(index=False))

=== Q4: Average Positions Gained Per Race (Finishers Only) ===
    driver_name  avg_grid  avg_finish  avg_positions_gained  races
   Sergio Perez      9.16        3.89                  5.26     19
   Lance Stroll     12.00        8.81                  3.19     16
   Yuki Tsunoda     14.00       11.21                  2.79     14
    Guanyu Zhou     15.75       13.00                  2.75     12
   Esteban Ocon     11.00        8.54                  2.46     13
 Max Verstappen      3.18        1.27                  1.91     22
 Lewis Hamilton      6.40        4.90                  1.50     20
   Pierre Gasly     11.11        9.68                  1.42     19
Valtteri Bottas     13.17       11.83                  1.33     12
   Lando Norris      7.56        6.28                  1.28     18
 George Russell      7.00        6.11                  0.89     18
  Oscar Piastri      8.36        7.93                  0.43     14
Fernando Alonso      6.00        5.60                  0.40     20

Query 5: Points progression per driver using CTE + window function (most complex)

In [25]:
q5 = pd.read_sql_query("""
    WITH cumulative_points_calc AS (
        SELECT driver_name,
               race_date,
               track,
               points,
               SUM(points) OVER (
                   PARTITION BY driver_name
                   ORDER BY race_date
                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
               ) AS cumulative_points
        FROM f1
    ),
    ranked_cumulative AS (
        SELECT driver_name,
               race_date,
               track,
               points,
               cumulative_points,
               RANK() OVER (
                   PARTITION BY race_date
                   ORDER BY cumulative_points DESC
               ) AS championship_rank
        FROM cumulative_points_calc
    )
    SELECT driver_name, track, race_date, points, cumulative_points, championship_rank
    FROM ranked_cumulative
    WHERE driver_name IN ('Max Verstappen', 'Lewis Hamilton', 'Fernando Alonso', 'Sergio Perez')
    ORDER BY race_date, championship_rank
""", conn)

print("=== Q5: Cumulative Championship Points Progression (Top Drivers) ===")
print(q5.to_string(index=False))

=== Q5: Cumulative Championship Points Progression (Top Drivers) ===
    driver_name         track  race_date  points  cumulative_points  championship_rank
 Max Verstappen       Bahrain 2023-03-05      25                 25                  1
   Sergio Perez       Bahrain 2023-03-05      18                 18                  2
Fernando Alonso       Bahrain 2023-03-05      15                 15                  3
 Lewis Hamilton       Bahrain 2023-03-05      10                 10                  5
 Max Verstappen  Saudi Arabia 2023-03-19      19                 44                  1
   Sergio Perez  Saudi Arabia 2023-03-19      25                 43                  2
Fernando Alonso  Saudi Arabia 2023-03-19      15                 30                  3
 Lewis Hamilton  Saudi Arabia 2023-03-19      10                 20                  4
 Max Verstappen     Australia 2023-04-02      25                 69                  1
   Sergio Perez     Australia 2023-04-02      11             